In [1]:
# make_hrv_tables_from_stage1.py
# Combine per-file HRV CSVs (from Stage 1) into group summaries

from pathlib import Path
import pandas as pd
import numpy as np

# ---- Paths ----
PROC = Path("D:/Git_repository/result folder/data/processed")
HRV_ROOT = PROC / "hrv"
SUM_DIR  = PROC / "summaries"
SUM_DIR.mkdir(parents=True, exist_ok=True)

# ---- Groups ----
GROUPS = ["conventional", "vr_high", "vr_low", "stress"]

# ---- Collect all participant HRV CSVs ----
rows = []
for g in GROUPS:
    gdir = HRV_ROOT / g
    if not gdir.exists():
        print(f"⚠️ Skip {g}: folder not found")
        continue
    for f in gdir.glob("*.csv"):
        try:
            h = pd.read_csv(f)
        except Exception as e:
            print(f"⚠️ Failed {f.name}: {e}")
            continue
        if h.empty:
            continue
        h["participant"] = f.stem
        h["group"] = g
        rows.append(h)

if not rows:
    raise SystemExit("❌ No HRV CSVs found — check processed/hrv/* folders.")

# ---- Merge into one table ----
hrv_all = pd.concat(rows, ignore_index=True)
hrv_all.to_csv(SUM_DIR / "hrv_all_participants.csv", index=False)
print("✅ Saved:", SUM_DIR / "hrv_all_participants.csv")

# ---- Compute group mean / std ----
num_cols = hrv_all.select_dtypes(include=[np.number]).columns.tolist()

grp_mean = hrv_all.groupby("group")[num_cols].mean().reset_index()
grp_std  = hrv_all.groupby("group")[num_cols].std(ddof=1).reset_index()

grp_mean.to_csv(SUM_DIR / "hrv_group_mean.csv", index=False)
grp_std.to_csv(SUM_DIR / "hrv_group_std.csv", index=False)

print("✅ Saved:", SUM_DIR / "hrv_group_mean.csv")
print("✅ Saved:", SUM_DIR / "hrv_group_std.csv")

print("\nDone! You can now open these in 'view_hrv_results.py' to visualize.")


✅ Saved: D:\Git_repository\result folder\data\processed\summaries\hrv_all_participants.csv
✅ Saved: D:\Git_repository\result folder\data\processed\summaries\hrv_group_mean.csv
✅ Saved: D:\Git_repository\result folder\data\processed\summaries\hrv_group_std.csv

Done! You can now open these in 'view_hrv_results.py' to visualize.


In [2]:
import pandas as pd
from pathlib import Path

# ---- Paths ----
BASE = Path(r"D:/Git_repository/result folder/data/processed/summaries")

all_path  = BASE / "hrv_all_participants.csv"
mean_path = BASE / "hrv_group_mean.csv"
std_path  = BASE / "hrv_group_std.csv"

# ---- Load CSVs ----
hrv_all  = pd.read_csv(all_path)
hrv_mean = pd.read_csv(mean_path)
hrv_std  = pd.read_csv(std_path)

print("✅ Loaded hrv_all_participants.csv:", hrv_all.shape)
print(hrv_all.head(10))   # first 10 rows

print("\n✅ Loaded hrv_group_mean.csv:")
print(hrv_mean)

print("\n✅ Loaded hrv_group_std.csv:")
print(hrv_std)


✅ Loaded hrv_all_participants.csv: (94, 11)
     ok  n_beats  mean_HR_bpm    SDNN_s   RMSSD_s  pNN50_percent  LF_power  \
0  True      336    84.774976  0.062293  0.057779      35.347432  0.001532   
1  True      467   112.615823  0.150046  0.204129      51.776650  0.002970   
2  True      516   119.090478  0.173938  0.221484      77.272727  0.005492   
3  True      368    89.990386  0.202236  0.248643      71.142857  0.010401   
4  True      338    83.160786  0.179180  0.232700      72.477064  0.004933   
5  True      393    98.142217  0.141912  0.210249      36.387435  0.002585   
6  True      342    84.214990  0.145896  0.182688      46.666667  0.005589   
7  True      347    85.851987  0.190426  0.265361      60.960961  0.007612   
8  True      326    74.537445  0.183062  0.174567      54.804270  0.007869   
9  True      359    89.436326  0.053021  0.050158      16.853933  0.000548   

   HF_power  LF_HF_ratio  participant         group  
0  0.000526     2.909112          100  conv